# Tailscale in userspace mode, because there is no TUN

The NetBird probe measured `/dev/net/tun` absent and `ip tuntap add` refused, so a
kernel-WireGuard client cannot run here. Tailscale works in userspace:

```
tailscaled --tun=userspace-networking --socks5-server=127.0.0.1:1055
```

**The auth key is a secret.** It is read from Kaggle Secrets (`TS_AUTHKEY`) and is
never written into this notebook, its metadata, its output, or a log. If a key has
ever been pasted into a chat or a file, revoke it in the admin console and issue a
new one.

This kernel proves the overlay comes up and prints the node's tailnet address. It
serves no model.


In [ ]:
import os, shutil, subprocess, time
from pathlib import Path

def sh(cmd, timeout=120):
    r = subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout + r.stderr).strip()

try:
    from kaggle_secrets import UserSecretsClient
    TS_AUTHKEY = UserSecretsClient().get_secret("TS_AUTHKEY")
except Exception:
    TS_AUTHKEY = os.environ.get("TS_AUTHKEY", "")
if not TS_AUTHKEY:
    raise SystemExit("add TS_AUTHKEY to Kaggle Secrets (Add-ons -> Secrets); never paste it here")

# The vendor script is a download-and-run. Accept it explicitly, and record the URL.
rc, out = sh("curl -fsSL https://tailscale.com/install.sh -o /tmp/ts-install.sh && sha256sum /tmp/ts-install.sh && sh /tmp/ts-install.sh 2>&1 | tail -3", timeout=600)
print("install rc", rc); print(out[-600:])
assert shutil.which("tailscaled") and shutil.which("tailscale"), "tailscale did not install"

SOCK = "/tmp/ts.sock"
rc, out = sh(f"nohup tailscaled --tun=userspace-networking --socks5-server=127.0.0.1:1055 "
             f"--socket={SOCK} --state=/kaggle/working/ts.state >/tmp/tailscaled.log 2>&1 & sleep 6; "
             f"tailscale --socket={SOCK} status 2>&1 | head -5")
print("tailscaled rc", rc); print(out)

# The key goes through the environment of this one command, never through a file.
env_key = TS_AUTHKEY
rc, out = sh(f'tailscale --socket={SOCK} up --auth-key="$TSKEY" --hostname=kaggle-teacher --accept-dns=false',
             timeout=120) if False else (0, "")
proc = subprocess.run(["bash", "-lc", f"tailscale --socket={SOCK} up --auth-key=\"$TSKEY\" --hostname=kaggle-teacher --accept-dns=false"],
                      capture_output=True, text=True, env={**os.environ, "TSKEY": env_key}, timeout=180)
print("up rc", proc.returncode)
print((proc.stdout + proc.stderr).strip()[:600])

rc, ip = sh(f"tailscale --socket={SOCK} ip -4")
print("tailnet ipv4:", ip)
Path("/kaggle/working/tailscale.json").write_text(json.dumps({"ipv4": ip, "status_rc": rc}, indent=2))
print("PROBE: userspace tailscale came up" if ip.startswith("100.") else "PROBE: no tailnet address; see status above")
